# What is Binarization?

* This is same as `discretization`. The only difference is it converts the data into binary values i.e 0 or 1. <br>
* This is only used in some particular cases and not wverywhere.
* For eg, Suppose we have a grayscale image and its pixels ranging from 0-255 and we wanna convert this image into black and white.
* In this case we select a threshold value (255 / 2) = 127.5
* Each value <= 127.5 converts to 0 and value > 127.5 converts to 1

# Binarization takes two arguments, **threshold** and **copy**.

* **threshold**: The values less than thres, is converted to 0 and the values more than thres is converted to 1
* **copy**: copy is very simple. If set as True, it creates a new column for 0 and 1 values or if set to False, makes changes into the existing column.

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('00 titanic.csv', usecols = ['Age', "Fare", 'SibSp', 'Parch', 'Survived'])

In [3]:
df.sample(10)

,Survived,Age,SibSp,Parch,Fare
251,0,29.0,1,1,10.4625
33,0,66.0,0,0,10.5000
152,0,55.5,0,0,8.0500
409,0,NaN,3,1,25.4667
493,0,71.0,0,0,49.5042
699,0,42.0,0,0,7.6500
472,1,33.0,1,2,27.7500
269,1,35.0,0,0,135.6333
210,0,24.0,0,0,7.0500
867,0,31.0,0,0,50.4958


In [4]:
df['Family'] = df['SibSp'] + df['Parch']

In [5]:
df.columns.tolist()

['Survived', 'Age', 'SibSp', 'Parch', 'Fare', 'Family']

In [6]:
df.drop(columns = ['SibSp', 'Parch'], axis = 'columns', inplace = True)

In [7]:
df.isnull().sum()

Survived      0
Age         177
Fare          0
Family        0
dtype: int64

In [8]:
df.dropna(inplace = True)

In [9]:
df.isnull().sum()

Survived    0
Age         0
Fare        0
Family      0
dtype: int64

In [10]:
df.sample(10)

,Survived,Age,Fare,Family
508,0,28.0,22.5250,0
762,1,20.0,7.2292,0
207,1,26.0,18.7875,0
769,0,32.0,8.3625,0
635,1,28.0,13.0000,0
847,0,35.0,7.8958,0
689,1,15.0,211.3375,1
200,0,28.0,9.5000,0
835,1,39.0,83.1583,2
751,1,6.0,12.4750,1


# Train/Test split

In [11]:
x = df.drop(columns = 'Survived', axis = 'columns')
y = df.drop(x, axis = 'columns')

In [12]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2)

# Before Binarzation.

In [13]:
dt_b4 = DecisionTreeClassifier()

In [14]:
dt_b4.fit(x_train, y_train)

DecisionTreeClassifier()

In [15]:
pred_b4 = dt_b4.predict(x_test)

In [16]:
from sklearn.metrics import accuracy_score

acc_b4 = accuracy_score(y_test, pred_b4)

print(acc_b4)

0.5804195804195804


In [17]:
from sklearn.model_selection import cross_val_score

cross_val_score(estimator = dt_b4, X = x_train, y = y_train, cv = 10, scoring = 'accuracy').mean()

0.6463702359346643

# After Binarization.

In [18]:
from sklearn.preprocessing import Binarizer

In [19]:
data = [
    ('Family', Binarizer(copy = False), ['Family'])
]

ct = ColumnTransformer(transformers = data, remainder = 'passthrough')

In [20]:
x_train_trans = ct.fit_transform(x_train)
x_test_trans = ct.transform(x_test)

In [21]:
print(ct.named_transformers_)


{'Family': Binarizer(copy=False), 'remainder': 'passthrough'}


In [22]:
pd.DataFrame(x_train_trans, columns = ['Family', 'Age', "Fare"])

,Family,Age,Fare
0,0.0,40.0,153.4625
1,0.0,22.0,7.2292
2,1.0,50.0,247.5208
3,0.0,30.0,9.5000
4,0.0,25.0,7.7750
...,...,...,...
566,1.0,26.0,16.1000
567,1.0,10.0,27.9000
568,0.0,21.0,8.4333
569,0.0,26.0,10.5000


# After Binarizarion

In [23]:
dt_after = DecisionTreeClassifier()

In [24]:
dt_after.fit(x_train_trans, y_train)

DecisionTreeClassifier()

In [25]:
pred_after = dt_after.predict(x_test_trans)

In [26]:
acc_after = accuracy_score(y_test, pred_after)

acc_after

0.5734265734265734

In [27]:
cross_val_score(estimator = dt_after, X = x_train_trans, y = y_train, cv = 10, scoring = 'accuracy').mean()

0.6130066545674531